In [22]:
!pip install -U langgraph langchain langchain_openai tavily-python yfinance langchain-tavily langchain_community

import os
from typing import Annotated, TypedDict
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.tools import tool
import yfinance as yf
import pandas as pd # Import pandas for indicator calculations
import numpy as np # Import numpy for indicator calculations
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')


In [23]:
# --- 1. STATE DEFINITION ---

class TradingState(TypedDict):
    ticker: str
    technical_analysis: list[float] # Changed to list of floats for indicator calculation
    news_sentiment: str
    final_signal: str
    indicators: dict # New field to store calculated indicators
    trade_strategy: str # New field for entry/exit points
    user_approval: str # HITL feedback
    status: str

In [24]:
# --- 2. CUSTOM TOOLS ---
@tool
def get_stock_price(ticker: str):
    """Fetches the latest closing price and historical data for a ticker."""
    stock = yf.Ticker(ticker)
    # Fetch 90 days of history for indicator calculation
    hist = stock.history(period="90d")
    if hist.empty:
        return {"latest_price": None, "history_prices": [], "error": "No historical data found"}

    # Convert all history prices to standard Python floats to ensure msgpack serializability
    history_prices = [float(x) for x in hist['Close'].tolist()]

    return {"latest_price": hist['Close'].iloc[-1], "history_prices": history_prices}

search_tool = TavilySearch()

In [33]:

# --- 3. AGENTS ---
model = ChatOpenAI(model="gpt-4o-mini",api_key= OPENAI_API_KEY, temperature=0)

def technical_analyst(state: TradingState):
    price_data_dict = get_stock_price.invoke(state["ticker"])
    if price_data_dict.get("error"):
        return {"technical_analysis": [], "status": "error", "error_message": price_data_dict["error"]}
    return {"technical_analysis": price_data_dict["history_prices"], "status": "analyzing_news"}

def sentiment_analyst(state: TradingState):
    search_query = f"latest news sentiment for {state['ticker']} stock"
    news = search_tool.invoke(search_query)
    return {"news_sentiment": str(news), "status": "generating_signal"}

def generate_final_signal(state: TradingState):
    prompt = f"Based on Tech (last 90 days close): {state['technical_analysis']} and News: {state['news_sentiment']}, provide a final Buy/Sell/Hold recommendation. Provide a detailed explanation, broken down into multiple paragraphs."
    res = model.invoke(prompt)
    return {"final_signal": res.content, "status": "awaiting_strategy_approval"}
# Function to route based on user approval for strategy generation
def route_decision_for_strategy(state: TradingState):
    if state.get("user_approval") == "approved":
        return "trade_strategy_generator"
    else:
        return END

def trade_strategy_generator(state: TradingState):
    import pandas as pd
    import numpy as np

    close = pd.Series(state["technical_analysis"], dtype=float)
    last_close = float(close.iloc[-1])

    # =========================
    # Trend indicators
    # =========================
    sma_10 = float(close.rolling(10).mean().iloc[-1])
    sma_30 = float(close.rolling(30).mean().iloc[-1])

    # =========================
    # RSI
    # =========================
    delta = close.diff()

    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()

    if pd.isna(loss.iloc[-1]) or loss.iloc[-1] == 0:
        rsi = 50.0
    else:
        rs = gain.iloc[-1] / loss.iloc[-1]
        rsi = float(100 - (100 / (1 + rs)))

    # =========================
    # ATR (volatility)
    # needs high/low in state
    # fallback estimate if absent
    # =========================
    if "high" in state and "low" in state:
        high = pd.Series(state["high"], dtype=float)
        low = pd.Series(state["low"], dtype=float)

        tr1 = high - low
        tr2 = (high - close.shift()).abs()
        tr3 = (low - close.shift()).abs()

        true_range = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
        atr = float(true_range.rolling(14).mean().iloc[-1])
    else:
        atr = float(close.pct_change().std() * last_close * 2)

    # =========================
    # Market structure
    # =========================
    recent_high = float(close.tail(20).max())
    recent_low = float(close.tail(20).min())

    trend = "BULLISH" if sma_10 > sma_30 else "BEARISH"

    if rsi > 70:
        rsi_status = "OVERBOUGHT"
    elif rsi < 30:
        rsi_status = "OVERSOLD"
    else:
        rsi_status = "NEUTRAL"

    indicators = {
        "last_close": last_close,
        "sma_10": sma_10,
        "sma_30": sma_30,
        "rsi": rsi,
        "atr": atr,
        "recent_high": recent_high,
        "recent_low": recent_low,
    }

    prompt = f"""
You are a Senior Quantitative Trader at a hedge fund.

Analyze the following setup and generate a professional trading plan.

MARKET DATA
Ticker: {state["ticker"]}
Initial Signal: {state["final_signal"]}

Current Price: {last_close:.2f}

TREND
- SMA 10: {sma_10:.2f}
- SMA 30: {sma_30:.2f}
- Trend Condition: {trend}

MOMENTUM
- RSI(14): {rsi:.2f}
- RSI Status: {rsi_status}

VOLATILITY
- ATR(14): {atr:.2f}

MARKET STRUCTURE
- Recent 20-bar High: {recent_high:.2f}
- Recent 20-bar Low: {recent_low:.2f}

TRADING FRAMEWORK
1) ENTRY
- Avoid chasing price extremes.
- For bullish setups:
  prefer pullback entries near SMA10 / support / breakout retest.
- For bearish setups:
  prefer rallies into SMA10 / resistance / breakdown retest.

2) STOP LOSS
- Must be placed at technical invalidation:
  below SMA30,
  beyond recent swing low/high,
  or beyond ATR volatility band.
- Explain why this level invalidates the setup.

3) TAKE PROFIT
- Define:
  TP1 = conservative target
  TP2 = primary objective
  TP3 = stretch target
- Base targets on:
  resistance/support,
  measured move,
  volatility expansion,
  breakout continuation potential.

4) RISK ASSESSMENT
Include:
- setup quality (High / Medium / Low)
- main risk to thesis
- what confirmation is needed before entry

TASK
Provide concise professional analysis with headings:

### Trade Bias
### Entry Zone
### Stop Loss
### Take Profit
### Risk Assessment
### Trade Thesis

Be precise with actual price levels.
Do not use generic percentage stops or arbitrary 2:1 reward/risk ratios.
Use market structure, volatility, and momentum.
"""

    res = model.invoke(prompt)

    return {"trade_strategy": res.content, "indicators": indicators, "status": "complete"}

In [35]:
# --- 4. GRAPH CONSTRUCTION ---
workflow = StateGraph(TradingState)

workflow.add_node("tech_analyst", technical_analyst)
workflow.add_node("news_analyst", sentiment_analyst)
workflow.add_node("generate_final_signal", generate_final_signal) # Signal always generated first
workflow.add_node("human_gate", lambda state: state) # HITL Pause Point after signal
workflow.add_node("trade_strategy_generator", trade_strategy_generator) # New node for strategy


workflow.set_entry_point("tech_analyst")
workflow.add_edge("tech_analyst", "news_analyst")
workflow.add_edge("news_analyst", "generate_final_signal")
workflow.add_edge("generate_final_signal", "human_gate") # Human intervenes after signal



workflow.add_conditional_edges("human_gate", route_decision_for_strategy)
workflow.add_edge("trade_strategy_generator", END) # End after generating the strategy

# Compile with Interruption
app = workflow.compile(checkpointer=MemorySaver(), interrupt_before=["human_gate"])

In [41]:
# --- 5. CORE FUNCTION ---
def execute_workflow(ticker: str, thread_id="trade_1"):
    config = {"configurable": {"thread_id": thread_id}}
    app.invoke({"ticker": ticker, "user_approval": "", "status": "starting", "indicators": {}, "trade_strategy": ""}, config)

    state = app.get_state(config).values
    print(f"\n[PAUSED] Analysis for {ticker} is ready.")
    print(f"Technical Analysis (last 5 days close): {state['technical_analysis'][-5:]}") # Display last 5 prices
    print(f"Final Signal: {state['final_signal']}\n")
    print("Please approve to generate detailed trade strategy (entry, stop loss, take profit).")

    feedback = input("Type 'approved' to get the strategy, or 'rejected' to stop: ")
    resume_workflow(feedback, thread_id)

In [39]:
def resume_workflow(feedback: str, thread_id="trade_1"):
    config = {"configurable": {"thread_id": thread_id}}

    # Update the state with the human's feedback
    app.update_state(config, {"user_approval": feedback}, as_node="human_gate")

    print(f"--- Resuming workflow with feedback: '{feedback}' ---")

    # Explicitly invoke the app from the human_gate node to finish the workflow.
    # We do NOT use stream_mode here to avoid verbose output from internal LangGraph logging.
    app.invoke(None, config=config)

    # Retrieve the final state after the workflow has resumed and completed
    final_state_snapshot = app.get_state(config).values

    if final_state_snapshot.get("trade_strategy"):
        print("\n==================================")
        print("✅ FINAL TRADE STRATEGY:")
        print("==================================")
        print(final_state_snapshot["trade_strategy"])
        print("==================================")
    elif final_state_snapshot.get("status") == "complete":
        print("\nWorkflow Finished (no strategy generated).")
    elif final_state_snapshot.get("final_signal"):
        print("\n==================================")
        print("✅ FINAL TRADE SIGNAL (Strategy Not Generated):")
        print("==================================")
        print(final_state_snapshot["final_signal"])
        print("==================================")

    else:
        print("\nWorkflow did not complete as expected or no final output was generated.")
        print(f"Current state: {final_state_snapshot}")

In [29]:
execute_workflow('GOLD')


[PAUSED] Analysis for GOLD is ready.
Technical Analysis (last 5 days close): [45.720001220703125, 45.16999816894531, 45.189998626708984, 42.630001068115234, 42.61000061035156]
Final Signal: Based on the recent performance of the Tech stock prices and the latest news sentiment regarding GOLD stock, a comprehensive analysis leads to a recommendation of **Hold** for Tech stocks and **Sell** for GOLD stocks. Below is a detailed breakdown of the reasoning behind these recommendations.

### Analysis of Tech Stock Performance

The Tech stock prices over the last 90 days show significant volatility, with a notable peak at approximately $63.92 followed by a decline to around $42.61. This fluctuation indicates a market that has experienced both bullish and bearish trends. The recent highs suggest that there was strong investor interest and potential growth, but the subsequent drop indicates a correction or a shift in market sentiment. 

The most recent prices hover around the mid-$40s, which is

### Test Case 1: Apple (AAPL) - Approved

In [31]:
# Simulate user input 'approved'
execute_workflow('AAPL', thread_id='trade_2') # User will input 'approved'


[PAUSED] Analysis for AAPL is ready.
Technical Analysis (last 5 days close): [270.7099914550781, 270.1700134277344, 271.3500061035156, 280.1400146484375, 276.05010986328125]
Final Signal: Based on the recent performance of Apple Inc. (AAPL) stock and the latest news sentiment, a recommendation can be made regarding whether to buy, sell, or hold the stock. 

### Stock Performance Analysis

Over the last 90 days, AAPL has shown a fluctuating trend, with a notable peak at around $280.14 and a recent closing price of approximately $276.05. The stock has experienced significant volatility, with prices dipping to around $246.47 before rebounding. This suggests that while there have been periods of decline, the stock has also demonstrated resilience and the ability to recover. The recent upward movement, particularly the increase of 3.43% noted in the news, indicates positive momentum, which could be a signal for potential buyers.

### Sentiment from Recent News

The news sentiment surroundi

\### Test Case 2: Tesla (TSLA) - Approved

In [36]:
# Simulate user input 'approved'
execute_workflow('TSLA', thread_id='trade_3') # User will input 'approved'


[PAUSED] Analysis for TSLA is ready.
Technical Analysis (last 5 days close): [376.0199890136719, 372.79998779296875, 381.6300048828125, 390.82000732421875, 391.1449890136719]
Final Signal: ### Recommendation: Hold

#### Market Performance Analysis
Over the last 90 days, Tesla's stock (TSLA) has exhibited significant volatility, with prices fluctuating from a high of approximately $485.56 to a low of around $346.65. The recent trend shows a downward trajectory, with the stock closing at $391.14, which is considerably lower than its peak. This decline suggests that the stock may be experiencing a correction after a period of rapid growth. The overall trend indicates that while there have been some recoveries, the stock has not maintained its previous highs, which could be a signal of underlying issues or market sentiment shifts.

#### Sentiment and News Analysis
Recent news surrounding Tesla has been mixed but leans towards a positive outlook. Reports highlight Tesla's ambitious growth 

### Test Case 3: Google (GOOG) - Approved


In [37]:
# Simulate user input 'approved'
execute_workflow('GOOG', thread_id='trade_4') # User will input 'approved'


[PAUSED] Analysis for GOOG is ready.
Technical Analysis (last 5 days close): [347.5, 347.30999755859375, 381.94000244140625, 383.2200012207031, 380.0400085449219]
Final Signal: Based on the recent performance of GOOG stock and the accompanying news sentiment, a **Hold** recommendation is warranted at this time. This conclusion is drawn from a combination of technical analysis of the stock's price movements over the last 90 days and the current sentiment reflected in the news articles.

### Technical Analysis

The price data for GOOG over the last 90 days shows a significant amount of volatility, with the stock fluctuating between a high of approximately $384.16 and a low of around $273.14. The recent trend indicates a recovery from lower levels, with the stock closing at $380.04, which is relatively close to its recent highs. The stock has shown resilience, bouncing back from lower price points, particularly in the last few weeks, where it has consistently closed above the $300 mark. 

### Test Case 4: Pfizer (PFE) - Reject

In [42]:
# Simulate user input 'reject'
execute_workflow('PFE', thread_id='trade_5') # User will input 'reject'


[PAUSED] Analysis for PFE is ready.
Technical Analysis (last 5 days close): [26.479999542236328, 26.260000228881836, 26.700000762939453, 26.329999923706055, 26.3799991607666]
Final Signal: Based on the recent performance of Pfizer Inc. (PFE) stock and the sentiment analysis of news articles related to the company, a recommendation can be made regarding whether to buy, sell, or hold the stock. 

### Stock Performance Analysis

Over the last 90 days, PFE stock has shown a generally upward trend, with prices fluctuating between approximately $24.47 and $28.55. The stock has recently closed at around $26.33, which is near the lower end of its recent trading range. The stock's performance indicates a recovery from lower levels, suggesting that it may be gaining momentum. The average price over the last 90 days appears to be around $26.00, which indicates that the current price is slightly above this average, suggesting a potential for further growth.

### News Sentiment Analysis

The senti

### Test Case 5: Bitcoin (BTC-USD) - Approved

In [43]:
# Simulate user input 'approved'
execute_workflow('BTC-USD', thread_id='trade_6') # User will input 'approved'


[PAUSED] Analysis for BTC-USD is ready.
Technical Analysis (last 5 days close): [76304.3203125, 78179.0, 78657.25, 78538.2265625, 80462.0]
Final Signal: Based on the recent price trends and sentiment analysis for Bitcoin (BTC-USD), a comprehensive recommendation can be formulated. The analysis will consider the last 90 days of closing prices, recent news sentiment, and overall market conditions.

### Price Trend Analysis

Over the last 90 days, Bitcoin has exhibited significant volatility, with prices fluctuating between approximately $62,221 and $80,462. The recent upward trend is notable, especially as Bitcoin has recently surpassed the $80,000 mark, reaching a three-month high. This upward movement suggests a bullish sentiment in the market, particularly in the last few weeks, where the price has shown a consistent increase. The average closing price over the last 90 days indicates a general upward trajectory, which is a positive sign for potential investors.

### Sentiment Analysi